# Unsupervised Learning and Clustering for Dog Growth Profiles

This notebook connects the third course topic, **Unsupervised Learning and Clustering**, to the Cane Corso Growth Intelligence project.

The goal is different from regression and classification:

- regression predicts a number, such as expected weight;
- classification predicts a known label, such as `normal_growth` or `needs_attention`;
- clustering discovers possible groups when no target label is given.

In the project story, clustering can help answer:

```text
Which growth records look naturally similar, even before a human label is assigned?
```

Responsible interpretation: clusters are **exploratory mathematical groups**, not veterinary diagnoses, not breed certification, and not official health categories.


## Mathematical Formulation

For clustering, the dataset is represented as a matrix:

```text
X = [x_1, x_2, ..., x_n]
```

Each row is a dog growth record and each column is a numeric feature.

In this notebook, the features include:

```text
visit_age_months
weight_kg
average_adult_breed_weight_kg
weight_to_adult_ratio
age_weight_ratio
adult_weight_group
```

There is no target variable:

```text
y = not used
```

The main distance idea is Euclidean distance:

```text
distance(a, b) = sqrt(sum((a_j - b_j)^2))
```

Scaling is required because features use different numeric units. Without scaling, one large-scale feature could dominate the distance calculation.


## Course Coverage in This Notebook

This notebook covers:

- Unsupervised Learning problem statement, intuition and challenges;
- K-Means Clustering motivation, example and `k-means++` initialization;
- Hierarchical Clustering motivation and example;
- comparison between K-Means and Hierarchical Clustering;
- DBSCAN for density-based clustering and outlier/noise detection.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import davies_bouldin_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dog_growth_public_sample.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)


## 1. Load the Processed Real Public Growth Sample

The notebook uses the processed public dog growth sample already included in the project.

The original raw dataset archive is not required for this notebook.


In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


## 2. Feature Preparation for Clustering

Clustering is distance-based, so feature preparation is especially important.

I create a small set of numeric features that describe each growth record:

- age in months;
- current weight;
- average adult breed weight;
- ratio between current weight and expected adult breed weight;
- age-to-weight relationship;
- rough adult breed weight group.

The sample is limited to keep the notebook fast and easy to run in a student environment.


In [ ]:
work = df.dropna(
    subset=["visit_age_months", "weight_kg", "average_adult_breed_weight_kg"]
).copy()

work = work[
    (work["visit_age_months"] >= 0)
    & (work["visit_age_months"] <= 84)
    & (work["weight_kg"] > 0)
    & (work["average_adult_breed_weight_kg"] > 0)
].copy()

work = work.sample(n=min(500, len(work)), random_state=RANDOM_STATE).reset_index(drop=True)

work["weight_to_adult_ratio"] = work["weight_kg"] / work["average_adult_breed_weight_kg"]
work["age_weight_ratio"] = work["weight_kg"] / (work["visit_age_months"] + 1)
work["adult_weight_group"] = pd.cut(
    work["average_adult_breed_weight_kg"],
    bins=[0, 10, 25, 45, 200],
    labels=[0, 1, 2, 3],
).astype(int)

features = [
    "visit_age_months",
    "weight_kg",
    "average_adult_breed_weight_kg",
    "weight_to_adult_ratio",
    "age_weight_ratio",
    "adult_weight_group",
]

X = work[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Clustering sample shape:", X.shape)
X.head()


## 3. Unsupervised Learning: Intuition and Challenges

In supervised learning, the model has a known answer during training:

```text
X -> y
```

In unsupervised learning, there is no known target label:

```text
X -> hidden structure
```

Main challenges:

- clusters may not have a real biological meaning;
- different scaling can change the result;
- different algorithms can produce different groups;
- the number of clusters is not always obvious;
- outliers can strongly affect distance-based methods.

For this project, the safe interpretation is:

```text
clustering suggests growth-pattern groups for exploration only
```


## 4. K-Means Clustering with k-means++

K-Means tries to place `k` centroids and assign each point to the nearest centroid.

The objective is to reduce within-cluster squared distance:

```text
minimize sum(distance(point, assigned_centroid)^2)
```

The `k-means++` initialization helps choose better starting centroids than random initialization. This can make the result more stable.


In [ ]:
kmeans_scores = []

for k in range(2, 7):
    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        random_state=RANDOM_STATE,
    )
    labels = model.fit_predict(X_scaled)
    kmeans_scores.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(X_scaled, labels),
            "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        }
    )

kmeans_scores_df = pd.DataFrame(kmeans_scores)
kmeans_scores_df


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(kmeans_scores_df["k"], kmeans_scores_df["inertia"], marker="o")
plt.title("K-Means Elbow Check")
plt.xlabel("Number of clusters k")
plt.ylabel("Inertia")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "kmeans_elbow_check.png", dpi=160)
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(kmeans_scores_df["k"], kmeans_scores_df["silhouette"], marker="o")
plt.title("K-Means Silhouette Score")
plt.xlabel("Number of clusters k")
plt.ylabel("Silhouette score")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "kmeans_silhouette_score.png", dpi=160)
plt.show()


### K-Means Example with 3 Clusters

I use 3 clusters as an easy educational example. The purpose is not to claim that exactly three true biological growth groups exist. The purpose is to show how K-Means groups records by similarity.


In [ ]:
kmeans = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=RANDOM_STATE)
work["kmeans_cluster"] = kmeans.fit_predict(X_scaled)

cluster_summary = work.groupby("kmeans_cluster")[features].mean().round(2)
cluster_summary["records"] = work["kmeans_cluster"].value_counts().sort_index().values
cluster_summary


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=work["kmeans_cluster"], alpha=0.75)
plt.title("K-Means Growth Pattern Groups - PCA View")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.grid(True, alpha=0.3)
plt.colorbar(scatter, label="K-Means cluster")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "kmeans_growth_pattern_groups_pca.png", dpi=160)
plt.show()


## 5. Hierarchical Clustering

Hierarchical clustering builds a hierarchy of groups.

In this notebook, I use agglomerative clustering:

```text
start with many small groups -> merge closest groups step by step
```

Motivation:

- useful when I want to understand nested structure;
- does not require centroids like K-Means;
- can be interpreted as a tree-like grouping process.

The downside is that it can be slower on large datasets.


In [ ]:
hierarchical_scores = []

for k in range(2, 7):
    hc = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = hc.fit_predict(X_scaled)
    hierarchical_scores.append(
        {
            "k": k,
            "silhouette": silhouette_score(X_scaled, labels),
            "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        }
    )

hierarchical_scores_df = pd.DataFrame(hierarchical_scores)
hierarchical_scores_df


In [ ]:
hc = AgglomerativeClustering(n_clusters=3, linkage="ward")
work["hierarchical_cluster"] = hc.fit_predict(X_scaled)

hierarchical_summary = work.groupby("hierarchical_cluster")[features].mean().round(2)
hierarchical_summary["records"] = work["hierarchical_cluster"].value_counts().sort_index().values
hierarchical_summary


In [ ]:
plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=work["hierarchical_cluster"], alpha=0.75)
plt.title("Hierarchical Clustering Growth Groups - PCA View")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.grid(True, alpha=0.3)
plt.colorbar(scatter, label="Hierarchical cluster")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "hierarchical_growth_groups_pca.png", dpi=160)
plt.show()


## 6. K-Means vs Hierarchical Clustering

The two methods can be compared with metrics, but the interpretation should remain careful.

- Silhouette score: higher is usually better.
- Davies-Bouldin score: lower is usually better.

These metrics help compare mathematical separation, but they do not prove that a cluster has medical meaning.


In [ ]:
comparison = pd.DataFrame(
    [
        {
            "method": "K-Means",
            "k": 3,
            "silhouette": silhouette_score(X_scaled, work["kmeans_cluster"]),
            "davies_bouldin": davies_bouldin_score(X_scaled, work["kmeans_cluster"]),
        },
        {
            "method": "Hierarchical Clustering",
            "k": 3,
            "silhouette": silhouette_score(X_scaled, work["hierarchical_cluster"]),
            "davies_bouldin": davies_bouldin_score(X_scaled, work["hierarchical_cluster"]),
        },
    ]
).round(4)

comparison


### Pros and Cons

| Method | Pros | Cons | Project interpretation |
|---|---|---|---|
| K-Means | fast, simple, easy cluster centers | must choose k, assumes rounded clusters, sensitive to scaling/outliers | good first grouping of growth profiles |
| Hierarchical Clustering | shows nested grouping idea, no centroid assumption | can be slower, still needs a cut level / number of groups | useful for exploring similarity structure |


## 7. DBSCAN

DBSCAN is density-based. It groups points that are close and dense, and it can mark sparse points as noise.

Important parameters:

- `eps`: neighborhood radius;
- `min_samples`: minimum number of nearby points required to form dense region.

This is useful because unusual records may appear as noise instead of being forced into a cluster.


In [ ]:
min_samples = 8
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X_scaled)
distances, _ = neighbors_fit.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(7, 4))
plt.plot(k_distances)
plt.title("DBSCAN k-distance Check")
plt.xlabel("Sorted records")
plt.ylabel(f"Distance to {min_samples}th neighbor")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dbscan_k_distance_check.png", dpi=160)
plt.show()


In [ ]:
dbscan = DBSCAN(eps=0.35, min_samples=min_samples)
work["dbscan_cluster"] = dbscan.fit_predict(X_scaled)

dbscan_cluster_count = len(set(work["dbscan_cluster"]) - {-1})
dbscan_noise_rate = (work["dbscan_cluster"] == -1).mean()

print("DBSCAN clusters excluding noise:", dbscan_cluster_count)
print("DBSCAN noise rate:", round(dbscan_noise_rate, 3))
work["dbscan_cluster"].value_counts().sort_index()


In [ ]:
plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=work["dbscan_cluster"], alpha=0.75)
plt.title("DBSCAN Density Groups and Noise - PCA View")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.grid(True, alpha=0.3)
plt.colorbar(scatter, label="DBSCAN cluster (-1 = noise)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dbscan_density_groups_pca.png", dpi=160)
plt.show()


## 8. Final Interpretation

The clustering stage adds a new type of machine-learning insight to the project:

```text
not prediction -> discovery of possible natural groups
```

Practical interpretation for Cane Corso Growth Intelligence:

- K-Means can provide simple growth-pattern groups;
- Hierarchical Clustering can show similarity structure;
- DBSCAN can flag records that do not fit dense groups;
- all clusters must remain exploratory and require human interpretation.

This completes the core requirements from the Unsupervised Learning and Clustering lecture while keeping the project connected to the same growth-monitoring story.
